# 09 — GraphRAG: Entity and Relationship Retrieval

## Goal
Build a small, local graph over benchmark chunks. An LLM extracts candidate entities and relationships from a sample of chunks; NetworkX stores the graph, and question terms are used to find related nodes and source chunks. The retrieved graph evidence and text excerpts are then given to the model.

This is a **lightweight educational GraphRAG pattern**, not Microsoft's GraphRAG package or a complete global-community-summary implementation. LLM-extracted edges can be wrong; inspect them before relying on them.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first.

## Architecture

```mermaid
flowchart TD
    A["Benchmark chunks"] --> B["Extract entities / relations"]
    B --> C["Build NetworkX graph"]
    Q["User question"] --> D["Find matching graph nodes"]
    C --> D
    D --> E["Collect linked source chunks"]
    E --> F["Combine graph facts + text"]
    F --> G["Grounded answer with sources"]

    classDef data fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef graph fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef query fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef evidence fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef answer fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A,B data;
    class C,D graph;
    class Q query;
    class E,F evidence;
    class G answer;
```

## 1. Imports and paths

In [ ]:
from pathlib import Path
import re
from collections import OrderedDict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vector_store").exists() and (PROJECT_ROOT.parent / "vector_store").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Open Chroma and install graph dependencies if needed

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Connected to:", COLLECTION_NAME)
print("Stored record count:", vector_store._collection.count())

# If NetworkX is missing, install it in your active notebook environment:
# %pip install networkx
import networkx as nx


## 4. Load a bounded set of source chunks

Graph extraction can be expensive. Start with a modest sample and increase it after inspecting quality. The code reads stored documents from the Chroma collection; it does not perform vector retrieval for graph construction.

In [ ]:
# Adjust this cap for experiments. The graph is built from these chunks only.
MAX_GRAPH_CHUNKS = 120

stored = vector_store.get(include=["documents", "metadatas"])
all_docs = []
for text, metadata in zip(stored.get("documents", []), stored.get("metadatas", [])):
    if text:
        from langchain_core.documents import Document
        all_docs.append(Document(page_content=text, metadata=metadata or {}))

graph_docs = all_docs[:MAX_GRAPH_CHUNKS]
print("Total stored chunks:", len(all_docs))
print("Chunks selected for graph extraction:", len(graph_docs))

## 5. Extract entities and relationships

The model is asked to return one JSON object with `entities` and `relationships`. Parsing is defensive because model output can be malformed. Each relationship records the chunk index that supports it.

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate

extract_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract a small knowledge graph from this Windows security benchmark excerpt. "
     "Return ONLY valid JSON with shape: "
     '{"entities":[{"name":"...","type":"..."}],'
     '"relationships":[{"source":"...","relation":"...","target":"..."}]}. '
     "Use only entities and relations explicitly supported by the excerpt. "
     "Keep names short. If none, return empty arrays."),
    ("human", "Source chunk index: {chunk_index}\\nExcerpt:\\n{text}")
])

def parse_json_object(raw: str) -> dict:
    # Strip common Markdown code fences if the model adds them.
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end <= start:
        raise ValueError("No JSON object found in model response")
    return json.loads(cleaned[start:end + 1])

def extract_graph_facts(text: str, chunk_index: int) -> dict:
    response = (extract_prompt | llm).invoke({
        "chunk_index": chunk_index, "text": text[:3500]
    })
    raw = response.content if hasattr(response, "content") else str(response)
    try:
        data = parse_json_object(raw)
    except Exception as exc:
        print(f"Skipping chunk {chunk_index}: could not parse extraction:", exc)
        return {"entities": [], "relationships": []}
    if not isinstance(data, dict):
        return {"entities": [], "relationships": []}
    return {
        "entities": data.get("entities", []) if isinstance(data.get("entities", []), list) else [],
        "relationships": data.get("relationships", []) if isinstance(data.get("relationships", []), list) else [],
    }

## 6. Build the graph

Nodes and edges retain `chunk_index` metadata so graph matches can be traced back to the source excerpt. Run this cell once; LLM extraction can take time depending on the model and chunk count.

In [ ]:
G = nx.MultiDiGraph()
extraction_log = []

for idx, doc in enumerate(graph_docs):
    facts = extract_graph_facts(doc.page_content, idx)
    extraction_log.append({"chunk_index": idx, **facts})
    for entity in facts["entities"]:
        if not isinstance(entity, dict) or not entity.get("name"):
            continue
        name = str(entity["name"]).strip()
        if not name:
            continue
        G.add_node(name, entity_type=str(entity.get("type", "unknown")))
    for rel in facts["relationships"]:
        if not isinstance(rel, dict):
            continue
        src, relation, tgt = rel.get("source"), rel.get("relation"), rel.get("target")
        if not src or not relation or not tgt:
            continue
        src, relation, tgt = str(src).strip(), str(relation).strip(), str(tgt).strip()
        G.add_node(src, entity_type="unknown")
        G.add_node(tgt, entity_type="unknown")
        G.add_edge(src, tgt, relation=relation, chunk_index=idx)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

## 7. Query the graph and retrieve supporting chunks

In [ ]:
def graph_retrieve(question: str, max_nodes: int = 8, max_chunks: int = 8) -> dict:
    q_terms = terms_for_graph(question)
    matched_nodes = []
    for node, attrs in G.nodes(data=True):
        node_terms = terms_for_graph(node)
        if q_terms & node_terms:
            matched_nodes.append(node)

    # Include immediate neighbors for relational context.
    expanded_nodes = set(matched_nodes)
    for node in matched_nodes:
        expanded_nodes.update(G.predecessors(node))
        expanded_nodes.update(G.successors(node))
    expanded_nodes = list(expanded_nodes)[:max_nodes]

    edges = []
    chunk_ids = set()
    for src, tgt, attrs in G.edges(data=True):
        if src in expanded_nodes or tgt in expanded_nodes:
            edges.append(f"{src} --[{attrs.get('relation', 'related to')}]--> {tgt} (chunk {attrs.get('chunk_index')})")
            if attrs.get("chunk_index") is not None:
                chunk_ids.add(attrs["chunk_index"])

    docs = [graph_docs[i] for i in sorted(chunk_ids) if 0 <= i < len(graph_docs)]
    # If graph matching is sparse, use vector search as a fallback.
    if not docs:
        docs = vector_store.similarity_search(question, k=5)
    return {
        "matched_nodes": matched_nodes[:max_nodes],
        "graph_facts": edges[:30],
        "documents": deduplicate_documents(docs)[:max_chunks],
    }

def terms_for_graph(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-zA-Z0-9]+", text.lower()) if len(t) > 2}

## 8. Generate an answer from graph facts and source text

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

graph_answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the graph facts and source excerpts below. "
     "Cite text evidence with labels such as [S1]. Graph edges are machine-extracted "
     "hypotheses, so do not treat them as more authoritative than the source excerpts. "
     "If evidence is insufficient, say so. Do not infer actual endpoint compliance."),
    ("human", "Question: {question}\\n\\nGraph facts:\\n{graph_facts}\\n\\nSource excerpts:\\n{context}")
])

def graph_rag(question: str) -> dict:
    retrieved = graph_retrieve(question)
    context = build_context(retrieved["documents"])
    response = (graph_answer_prompt | llm).invoke({
        "question": question,
        "graph_facts": "\\n".join(retrieved["graph_facts"]) or "[No graph facts matched.]",
        "context": context or "[No source excerpts retrieved.]",
    })
    answer = response.content if hasattr(response, "content") else str(response)
    return {**retrieved, "context": context, "answer": answer}

def display_graph_matches(result):
    print("Matched nodes:", result["matched_nodes"])
    print("Graph facts:")
    for fact in result["graph_facts"]:
        print(" -", fact)
    print("\\nAnswer:\\n", result["answer"])

## 9. Try a question

In [ ]:
question = "What recommendations relate to password policy?"
result = graph_rag(question)
display_graph_matches(result)

## 10. Inspect source chunks and graph size

In [ ]:
for i, doc in enumerate(result["documents"], start=1):
    print("=" * 88)
    print(source_label(doc, i))
    print(doc.page_content[:900])
    print()

print("Node count:", G.number_of_nodes())
print("Edge count:", G.number_of_edges())

## Summary and limitations

- Graph retrieval can surface relationships between entities that plain nearest-neighbor retrieval may miss.
- This notebook builds a graph from a bounded chunk sample, so graph coverage is incomplete.
- Entity and relation extraction can hallucinate or normalize terms inconsistently.
- Keyword matching is a simple graph-query baseline, not a full graph query language.
- The graph is in memory and is not persisted by default.
- For a stronger implementation, add extraction validation, entity resolution, provenance review, graph persistence, and systematic retrieval evaluation.
- The benchmark is a source of recommendations, not proof of current machine settings.